# Week 1 Assignment - MLOps - Iris Classifier using Vertex AI


## Overview

In this tutorial, we would load the IRIS Dataset and would try to create a basic classifdication model in the jupyter lab notebook inside GCP infrastructure.


### Dataset

This tutorial uses R.A. Fisher's Iris dataset, a small and popular dataset for machine learning experiments. Each instance has four numerical features, which are different measurements of a flower, and a target label that
categorizes the flower into: **Iris setosa**, **Iris versicolour** and **Iris virginica**.

This tutorial uses [a version of the Iris dataset available in the
scikit-learn library](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_iris.html#sklearn.datasets.load_iris).

### Costs

This tutorial uses billable components of Google Cloud:

* Vertex AI
* Cloud Storage

Learn about [Vertex AI
pricing](https://cloud.google.com/vertex-ai/pricing), [Cloud Storage
pricing](https://cloud.google.com/storage/pricing), 

In [18]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

## Get started

### Install Vertex AI SDK for Python and other required packages



In [19]:

# Vertex SDK for Python
! pip3 install --upgrade --quiet  google-cloud-aiplatform

### Set Google Cloud project information
Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [20]:
PROJECT_ID = "mlops-iitmadras"  # @param {type:"string"}
LOCATION = "us-central1"  # @param {type:"string"}

### Create a Cloud Storage bucket

Create a storage bucket to store intermediate artifacts such as datasets.

In [21]:
BUCKET_URI = f"gs://mlops-assignment-mlops-iitmadras-trialrun"  # @param {type:"string"}

**If your bucket doesn't already exist**: Run the following cell to create your Cloud Storage bucket.

In [22]:
! gsutil mb -l {LOCATION} -p {PROJECT_ID} {BUCKET_URI}

Creating gs://mlops-assignment-mlops-iitmadras-trialrun/...
ServiceException: 409 A Cloud Storage bucket named 'mlops-assignment-mlops-iitmadras-trialrun' already exists. Try another name. Bucket names must be globally unique across all Google Cloud projects, including those outside of your organization.


### Initialize Vertex AI SDK for Python

To get started using Vertex AI, you must have an existing Google Cloud project and [enable the Vertex AI API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com).

In [23]:
from google.cloud import aiplatform

aiplatform.init(project=PROJECT_ID, location=LOCATION, staging_bucket=BUCKET_URI)

### Import the required libraries

In [24]:
import os
import sys

In [25]:
DATASET_FOLDER_URI = f"{BUCKET_URI}/data/raw"

### Copying Data from the storage bucket

In [26]:
# !gsutil cp -r {DATA_FOLDER_URI} .

### Loading the data and splitiing into training and eval sets

In [27]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

data = pd.read_csv(f"{DATASET_FOLDER_URI}/iris.csv")
train_data, eval_data = train_test_split(data, test_size = 0.3, stratify = data['species'], random_state = 42)
train_data.to_csv(f"{DATASET_FOLDER_URI}/train_set.csv", index=False)
eval_data.to_csv(f"{DATASET_FOLDER_URI}/eval_set.csv", index=False)

## Simple Decision Tree model
Build a Decision Tree model on iris data

In [28]:
from pandas.plotting import parallel_coordinates
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn import metrics

data = pd.read_csv(f"{DATASET_FOLDER_URI}/train_set.csv")
data.head(5)

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,2.5,3.0,1.1,versicolor
1,6.2,2.2,4.5,1.5,versicolor
2,5.1,3.8,1.5,0.3,setosa
3,6.8,3.2,5.9,2.3,virginica
4,5.7,2.8,4.1,1.3,versicolor


In [29]:
train, test = train_test_split(data, test_size = 0.4, stratify = data['species'], random_state = 42)
X_train = train[['sepal_length','sepal_width','petal_length','petal_width']]
y_train = train.species
X_test = test[['sepal_length','sepal_width','petal_length','petal_width']]
y_test = test.species

In [30]:
mod_dt = DecisionTreeClassifier(max_depth = 3, random_state = 1)
mod_dt.fit(X_train,y_train)
prediction=mod_dt.predict(X_test)
accuracy_score = metrics.accuracy_score(prediction,y_test)
print('The accuracy of the Decision Tree is',"{:.3f}".format(accuracy_score))

The accuracy of the Decision Tree is 0.976


In [31]:
!mkdir artifacts

mkdir: cannot create directory ‘artifacts’: File exists


### Upload model artifacts and metrics to gcp storage with folders organized by their training execution timestamp

In [33]:
import pandas as pd
from datetime import datetime
import pickle
import joblib

joblib.dump(mod_dt, "artifacts/model.joblib")

timestamp = datetime.now().strftime("%Y%m%d-%H%M")
bucket_path = f"{BUCKET_URI}/train_runs/{timestamp}"

print(f"Artifacts for this run will be saved to: {bucket_path}")

# Save Log
log_msg = f"Training Accuracy: {accuracy_score} for the training run at {timestamp}"
!echo "{log_msg}" > training_log.txt
!gsutil cp training_log.txt {bucket_path}/training_log.txt

# Save model
!gsutil cp artifacts/model.joblib {bucket_path}/model.joblib
print("model saved successfully")

Artifacts for this run will be saved to: gs://mlops-assignment-mlops-iitmadras-trialrun/train_runs/20260217-0731
Copying file://training_log.txt [Content-Type=text/plain]...
/ [1 files][   76.0 B/   76.0 B]                                                
Operation completed over 1 objects/76.0 B.                                       
Copying file://artifacts/model.joblib [Content-Type=application/octet-stream]...
/ [1 files][  2.5 KiB/  2.5 KiB]                                                
Operation completed over 1 objects/2.5 KiB.                                      
model saved successfully
